In [ ]:
```xml
<VSCode.Cell language="markdown">
# AI-Specific Networking — Azure Supplement

This notebook demonstrates AI-specific networking on Azure cloud infrastructure:
1. Azure credentials setup (template)
2. Azure ND-series VMs (InfiniBand-enabled A100/H100)
3. Multi-node distributed inference with Llama-2-70B
4. Bandwidth profiling with NCCL tests
5. Cost analysis (NVLink VMs vs standard GPU VMs)
6. Topology visualization

**Azure VM families covered**:
- **NDv4-series**: 8× A100 80GB with NVLink + InfiniBand (200 Gb/s)
- **NDv5-series**: 8× H100 80GB with NVSwitch + InfiniBand (400 Gb/s)
- **NC-series**: Standard GPUs (V100, A100) without InfiniBand (PCIe-only)

**Prerequisites**:
- Azure subscription with quota for ND-series VMs
- Azure CLI installed (`az login`)
- SSH access to provisioned VMs

**Cost warning**: ND-series VMs are expensive ($20-$40/hour). Use spot instances or destroy resources immediately after testing.
</VSCode.Cell>

<VSCode.Cell language="python">
# Cell 1: Azure credentials (empty template — user must fill in)

"""
Azure Credentials Setup

Before running this notebook, you need:
1. Azure subscription with ND-series VM quota (request increase if needed)
2. Azure CLI installed and authenticated (run: az login)
3. Resource group and region selected

Fill in your values below:
"""

# ============================================================================
# USER CONFIGURATION — FILL THESE IN
# ============================================================================

AZURE_SUBSCRIPTION_ID = ""  # Your Azure subscription ID
AZURE_RESOURCE_GROUP = ""   # Resource group name (e.g., "inferencebase-gpu-rg")
AZURE_REGION = ""           # Azure region (e.g., "eastus", "westus2")

# VM Configuration
AZURE_VM_SKU = "Standard_ND96asr_v4"  # 8× A100 80GB with IB (change as needed)
AZURE_VM_COUNT = 2                     # Number of VMs for multi-node testing
AZURE_VM_NAME_PREFIX = "gpu-node"      # VM name prefix (will append -0, -1, etc.)

# SSH Configuration
AZURE_SSH_KEY_PATH = "~/.ssh/id_rsa.pub"  # Path to SSH public key
AZURE_ADMIN_USERNAME = "azureuser"         # VM admin username

# ============================================================================
# VALIDATION
# ============================================================================

import os
import sys

# Check if credentials are filled
if not AZURE_SUBSCRIPTION_ID:
    print("❌ ERROR: AZURE_SUBSCRIPTION_ID not set!")
    print("   → Get your subscription ID: az account show --query id -o tsv")
    sys.exit(1)

if not AZURE_RESOURCE_GROUP:
    print("❌ ERROR: AZURE_RESOURCE_GROUP not set!")
    print("   → Create a resource group: az group create --name <name> --location <region>")
    sys.exit(1)

if not AZURE_REGION:
    print("❌ ERROR: AZURE_REGION not set!")
    print("   → Choose a region with ND-series availability (eastus, westus2, etc.)")
    sys.exit(1)

# Check Azure CLI
try:
    import subprocess
    result = subprocess.run(['az', 'account', 'show'], capture_output=True, text=True, timeout=10)
    if result.returncode != 0:
        print("❌ ERROR: Azure CLI not authenticated")
        print("   → Run: az login")
        sys.exit(1)
    else:
        print("✅ Azure CLI authenticated")
        print(f"   Subscription: {AZURE_SUBSCRIPTION_ID[:8]}...")
        print(f"   Resource Group: {AZURE_RESOURCE_GROUP}")
        print(f"   Region: {AZURE_REGION}")
except FileNotFoundError:
    print("❌ ERROR: Azure CLI not installed")
    print("   → Install: https://docs.microsoft.com/cli/azure/install-azure-cli")
    sys.exit(1)
except subprocess.TimeoutExpired:
    print("⚠️  WARNING: Azure CLI check timed out")

print("\n✅ Configuration validated — ready to proceed")
print("\n⚠️  COST WARNING:")
print(f"   {AZURE_VM_SKU} costs ~$27/hour per VM")
print(f"   {AZURE_VM_COUNT} VMs = ~${AZURE_VM_COUNT * 27}/hour")
print(f"   Remember to DELETE resources after testing!")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## Azure ND-Series VMs: InfiniBand-Enabled GPU Clusters

Azure ND-series VMs are purpose-built for AI/ML workloads requiring high GPU-to-GPU bandwidth:

| VM SKU | GPUs | VRAM | NVLink | InfiniBand | CPU | RAM | Cost |
|--------|------|------|--------|------------|-----|-----|------|
| **Standard_ND96asr_v4** | 8× A100 80GB | 640 GB | 600 GB/s | 200 Gb/s HDR | 96 vCPU | 900 GB | ~$27/hr |
| **Standard_ND96amsr_A100_v4** | 8× A100 80GB | 640 GB | 600 GB/s | 200 Gb/s HDR | 96 vCPU | 1900 GB | ~$32/hr |
| **Standard_ND96isr_H100_v5** | 8× H100 80GB | 640 GB | 900 GB/s | 400 Gb/s NDR | 96 vCPU | 1900 GB | ~$42/hr |

**Key features**:
- **NVLink**: Full 8-GPU NVLink mesh within each VM
- **InfiniBand**: 200-400 Gb/s RDMA fabric for cross-node communication
- **GPUDirect RDMA**: Zero-copy GPU-to-GPU transfers across nodes
- **Low-latency NUMA**: Optimized CPU-GPU memory paths

**Comparison vs standard NC-series** (PCIe-only):
- NC-series (e.g., NC24ads_A100_v4): 2× A100, PCIe only, no InfiniBand → $7/hr
- ND-series: 8× A100, NVLink + InfiniBand → $27/hr (4× cost, 50× bandwidth)
</VSCode.Cell>

<VSCode.Cell language="python">
# Cell 2-3: Azure ND-series VMs (InfiniBand-enabled)

print("=== Provisioning Azure ND-Series VMs with InfiniBand ===\n")

import subprocess
import json
import time

# Define provisioning parameters
vm_config = {
    "vm_size": AZURE_VM_SKU,
    "count": AZURE_VM_COUNT,
    "resource_group": AZURE_RESOURCE_GROUP,
    "location": AZURE_REGION,
    "admin_username": AZURE_ADMIN_USERNAME,
    "ssh_key_path": os.path.expanduser(AZURE_SSH_KEY_PATH)
}

print(f"VM Configuration:")
print(f"  - SKU: {vm_config['vm_size']}")
print(f"  - Count: {vm_config['count']}")
print(f"  - Region: {vm_config['location']}")
print(f"  - Resource Group: {vm_config['resource_group']}")
print(f"\n⚠️  This will incur charges! Estimated cost: ~${AZURE_VM_COUNT * 27}/hour\n")

# Check if VMs already exist
print("Checking for existing VMs...")
try:
    result = subprocess.run(
        ['az', 'vm', 'list', 
         '--resource-group', AZURE_RESOURCE_GROUP,
         '--query', '[].name',
         '-o', 'json'],
        capture_output=True,
        text=True,
        timeout=30
    )
    
    if result.returncode == 0:
        existing_vms = json.loads(result.stdout) if result.stdout.strip() else []
        target_vms = [f"{AZURE_VM_NAME_PREFIX}-{i}" for i in range(AZURE_VM_COUNT)]
        
        if any(vm in existing_vms for vm in target_vms):
            print(f"✅ VMs already exist: {[vm for vm in target_vms if vm in existing_vms]}")
            print("   Skipping provisioning (delete VMs to re-provision)\n")
            vms_exist = True
        else:
            print(f"❌ No existing VMs found")
            print(f"   Would provision: {target_vms}")
            print(f"\n📝 To provision VMs, run:\n")
            
            for i in range(AZURE_VM_COUNT):
                vm_name = f"{AZURE_VM_NAME_PREFIX}-{i}"
                print(f"az vm create \\")
                print(f"  --resource-group {AZURE_RESOURCE_GROUP} \\")
                print(f"  --name {vm_name} \\")
                print(f"  --location {AZURE_REGION} \\")
                print(f"  --size {AZURE_VM_SKU} \\")
                print(f"  --image microsoft-dsvm:ubuntu-hpc:2004:latest \\")
                print(f"  --admin-username {AZURE_ADMIN_USERNAME} \\")
                print(f"  --ssh-key-values {vm_config['ssh_key_path']} \\")
                print(f"  --accelerated-networking true \\")
                print(f"  --priority Spot \\")  # Use spot for cost savings
                print(f"  --max-price -1 \\")   # Accept any spot price
                print(f"  --eviction-policy Deallocate\n")
            
            vms_exist = False
    else:
        print(f"⚠️  Error checking VMs: {result.stderr}")
        vms_exist = False

except subprocess.TimeoutExpired:
    print("⚠️  Timeout checking VMs")
    vms_exist = False
except json.JSONDecodeError:
    print("⚠️  Error parsing VM list")
    vms_exist = False

# Get VM IPs if they exist
if vms_exist:
    print("Fetching VM IP addresses...")
    vm_ips = {}
    
    for i in range(AZURE_VM_COUNT):
        vm_name = f"{AZURE_VM_NAME_PREFIX}-{i}"
        try:
            result = subprocess.run(
                ['az', 'vm', 'show',
                 '--resource-group', AZURE_RESOURCE_GROUP,
                 '--name', vm_name,
                 '--show-details',
                 '--query', 'publicIps',
                 '-o', 'tsv'],
                capture_output=True,
                text=True,
                timeout=15
            )
            
            if result.returncode == 0 and result.stdout.strip():
                vm_ips[vm_name] = result.stdout.strip()
                print(f"  {vm_name}: {vm_ips[vm_name]}")
            else:
                print(f"  {vm_name}: (no public IP)")
        except subprocess.TimeoutExpired:
            print(f"  {vm_name}: (timeout)")
    
    print(f"\n✅ {len(vm_ips)} VMs ready")
    print(f"\nTo SSH into VMs:")
    for vm_name, ip in vm_ips.items():
        print(f"  ssh {AZURE_ADMIN_USERNAME}@{ip}")
else:
    print("\n⚠️  VMs not provisioned in this session")
    print("   To proceed with this notebook, provision VMs first")

print("\n" + "="*80)
</VSCode.Cell>

<VSCode.Cell language="markdown">
## Multi-Node Distributed Inference: Llama-2-70B Across 2 Nodes

With ND-series VMs provisioned, we can now test multi-node tensor parallelism:
- Node 0: 4× A100 (ranks 0-3)
- Node 1: 4× A100 (ranks 4-7)
- Total: 8× A100 across 2 nodes connected via InfiniBand

This requires:
1. PyTorch Distributed setup (NCCL backend with InfiniBand)
2. NCCL environment variables configured
3. InfiniBand drivers installed (MLNX_OFED)
4. GPUDirect RDMA enabled
</VSCode.Cell>

<VSCode.Cell language="python">
# Cell 4-5: Multi-node distributed inference

print("=== Multi-Node Distributed Inference Setup ===\n")

print("🔧 Prerequisites for multi-node inference:")
print("""
1. MLNX_OFED Drivers (Mellanox OpenFabrics):
   - Install on all nodes: sudo apt install -y ibverbs-utils
   - Verify: ibstat (should show InfiniBand adapter active)

2. GPUDirect RDMA:
   - Load nvidia-peermem module: sudo modprobe nvidia-peermem
   - Verify: lsmod | grep nvidia_peermem

3. NCCL with InfiniBand support:
   - Included in PyTorch distributions for Azure ND-series
   - Verify: python -c "import torch; print(torch.cuda.nccl.version())"

4. Network configuration:
   - All nodes must be on same virtual network
   - NSG rules must allow InfiniBand ports (unrestricted within VNet)
   - Verify connectivity: ping <other-node-private-ip>
""")

# Sample PyTorch distributed setup script
setup_script = """
#!/bin/bash
# File: setup_distributed.sh (run on all nodes)

# NCCL configuration for InfiniBand
export NCCL_IB_DISABLE=0           # Enable InfiniBand
export NCCL_IB_HCA=mlx5_0          # InfiniBand adapter (verify with ibstat)
export NCCL_IB_GID_INDEX=3         # RoCE mode (if using Ethernet fallback)
export NCCL_NET_GDR_LEVEL=5        # Enable GPUDirect RDMA
export NCCL_SOCKET_IFNAME=eth0     # Primary network interface
export NCCL_DEBUG=INFO             # Verbose logging for debugging

# PyTorch distributed configuration
export MASTER_ADDR=<node-0-private-ip>  # Replace with actual IP
export MASTER_PORT=29500
export WORLD_SIZE=8                      # Total GPUs (2 nodes × 4 GPUs)
export RANK=<node-rank>                  # 0 for node-0, 1 for node-1
export LOCAL_RANK=<gpu-local-rank>       # 0-3 for GPUs on this node

# Launch distributed training/inference
python -m torch.distributed.launch \\
    --nproc_per_node=4 \\
    --nnodes=2 \\
    --node_rank=$RANK \\
    --master_addr=$MASTER_ADDR \\
    --master_port=$MASTER_PORT \\
    multi_node_inference.py
"""

print("📝 Sample setup script (save as setup_distributed.sh):")
print(setup_script)

# Sample Python inference script
inference_script = """
# File: multi_node_inference.py

import torch
import torch.distributed as dist
import os

def setup_distributed():
    # Initialize distributed process group
    dist.init_process_group(
        backend='nccl',
        init_method='env://',  # Use environment variables
        world_size=int(os.environ['WORLD_SIZE']),
        rank=int(os.environ['RANK'])
    )
    
    # Set local GPU
    local_rank = int(os.environ['LOCAL_RANK'])
    torch.cuda.set_device(local_rank)
    
    return dist.get_rank(), dist.get_world_size(), local_rank

def main():
    rank, world_size, local_rank = setup_distributed()
    
    print(f"[Rank {rank}/{world_size}] Running on GPU {local_rank}")
    print(f"[Rank {rank}] NCCL version: {torch.cuda.nccl.version()}")
    
    # Create dummy model (replace with actual Llama-2-70B)
    model = torch.nn.Linear(4096, 4096).to(f'cuda:{local_rank}')
    
    # Wrap with DistributedDataParallel (or use tensor parallelism)
    model = torch.nn.parallel.DistributedDataParallel(
        model,
        device_ids=[local_rank],
        output_device=local_rank
    )
    
    # Benchmark all-reduce (simulate gradient sync)
    tensor = torch.randn(1000, 1000, device=f'cuda:{local_rank}')
    
    # Warm-up
    for _ in range(5):
        dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    
    start.record()
    for _ in range(100):
        dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    end.record()
    
    torch.cuda.synchronize()
    elapsed_ms = start.elapsed_time(end) / 100
    
    if rank == 0:
        tensor_size_mb = tensor.numel() * tensor.element_size() / 1e6
        bandwidth_gb_s = (tensor_size_mb / 1000) / (elapsed_ms / 1000)
        print(f"\\n=== NCCL All-Reduce Benchmark ===")
        print(f"Tensor size: {tensor_size_mb:.2f} MB")
        print(f"All-reduce time: {elapsed_ms:.3f} ms")
        print(f"Effective bandwidth: {bandwidth_gb_s:.2f} GB/s")
    
    dist.destroy_process_group()

if __name__ == '__main__':
    main()
"""

print("\n📝 Sample inference script (save as multi_node_inference.py):")
print(inference_script)

print("\n" + "="*80)
print("To run multi-node inference:")
print("  1. Copy scripts to all nodes (scp or git clone)")
print("  2. Run setup_distributed.sh on each node (different RANK for each)")
print("  3. Monitor NCCL logs for topology detection and bandwidth")
print("="*80)
</VSCode.Cell>

<VSCode.Cell language="markdown">
## Bandwidth Profiling with NCCL Tests

NVIDIA provides official NCCL tests to benchmark collective operations (all-reduce, all-gather, etc.) across multi-GPU and multi-node setups. This is the standard tool for validating InfiniBand and NVLink performance.
</VSCode.Cell>

<VSCode.Cell language="python">
# Cell 6: Bandwidth profiling with NCCL tests

print("=== NCCL Tests for Bandwidth Profiling ===\n")

print("🔧 Installing and running NCCL tests:")
print("""
# On each Azure ND-series VM:

# 1. Clone NCCL tests
git clone https://github.com/NVIDIA/nccl-tests.git
cd nccl-tests

# 2. Build (requires CUDA and NCCL)
make MPI=1 MPI_HOME=/usr/lib/x86_64-linux-gnu/openmpi

# 3. Run all-reduce bandwidth test (single node, 8 GPUs)
./build/all_reduce_perf -b 1M -e 4G -f 2 -g 8

# Expected output (8× A100 with NVLink):
#   Size    Bandwidth (GB/s)
#   1 MB    ~50 GB/s
#   10 MB   ~400 GB/s
#   100 MB  ~500 GB/s
#   1 GB    ~520 GB/s  (close to theoretical 600 GB/s)
#   4 GB    ~550 GB/s

# 4. Run multi-node test (2 nodes × 4 GPUs = 8 GPUs total)
mpirun -np 8 \\
    -H node-0-private-ip:4,node-1-private-ip:4 \\
    -x NCCL_IB_DISABLE=0 \\
    -x NCCL_NET_GDR_LEVEL=5 \\
    -x NCCL_DEBUG=INFO \\
    ./build/all_reduce_perf -b 1M -e 4G -f 2 -g 4

# Expected cross-node bandwidth (InfiniBand HDR 200 Gb/s = 25 GB/s):
#   Within-node (NVLink): 400-550 GB/s
#   Cross-node (InfiniBand): 20-24 GB/s
#   Mixed: depends on NCCL's ring algorithm routing
""")

# Simulated NCCL test results
import pandas as pd

nccl_results = pd.DataFrame({
    'Message Size': ['1 MB', '10 MB', '100 MB', '1 GB', '4 GB'],
    'Single Node (NVLink)': [52, 387, 489, 523, 548],  # GB/s
    'Multi-Node (IB+NVLink)': [18, 142, 221, 237, 242],  # GB/s
    'PCIe-only (no IB)': [8, 31, 47, 51, 52]  # GB/s (for comparison)
})

print("\nSimulated NCCL All-Reduce Bandwidth Results:")
print(nccl_results.to_string(index=False))

print("\n📊 Key insights:")
print("  ✅ NVLink within-node: 500-550 GB/s (90-95% of theoretical 600 GB/s)")
print("  ✅ InfiniBand cross-node: 230-240 GB/s (mixed within+cross node)")
print("  ❌ PCIe-only: 50 GB/s (10× slower than NVLink)")
print("\n  → Conclusion: InfiniBand is 5× faster than PCIe for multi-node")
print("               but still 2× slower than NVLink within-node")

print("\n" + "="*80)
</VSCode.Cell>

<VSCode.Cell language="markdown">
## Cost Analysis: NVLink VMs vs Standard GPU VMs

Azure ND-series VMs with NVLink and InfiniBand are expensive. Let's compare total cost of ownership (TCO) against standard NC-series VMs (PCIe-only) for InferenceBase's Llama-2-70B deployment.
</VSCode.Cell>

<VSCode.Cell language="python">
# Cell 7: Cost analysis (NVLink VMs vs standard)

print("=== Azure GPU VM Cost Analysis ===\n")

# Define VM configurations
vm_configs = {
    "ND96asr_v4 (8× A100 NVLink+IB)": {
        "hourly_cost": 27.20,
        "gpus": 8,
        "gpu_type": "A100 80GB",
        "nvlink": True,
        "infiniband": True,
        "bandwidth_within_node": 600,  # GB/s
        "bandwidth_cross_node": 24,    # GB/s
        "min_vms_for_70b": 1,  # 70B INT4 = 35 GB, fits in 8× 80GB
        "latency_estimate_ms": 1800    # With NVLink
    },
    
    "NC96ads_A100_v4 (4× A100 PCIe)": {
        "hourly_cost": 15.00,
        "gpus": 4,
        "gpu_type": "A100 80GB",
        "nvlink": False,
        "infiniband": False,
        "bandwidth_within_node": 16,   # GB/s (PCIe)
        "bandwidth_cross_node": 0,     # N/A
        "min_vms_for_70b": 1,  # 70B INT4 fits, but latency poor
        "latency_estimate_ms": 5200    # With PCIe bottleneck
    },
    
    "NC24ads_A100_v4 (2× A100 PCIe)": {
        "hourly_cost": 7.35,
        "gpus": 2,
        "gpu_type": "A100 80GB",
        "nvlink": False,
        "infiniband": False,
        "bandwidth_within_node": 16,   # GB/s (PCIe)
        "bandwidth_cross_node": 0,     # N/A
        "min_vms_for_70b": 2,  # Need 2 VMs (4 GPUs total)
        "latency_estimate_ms": 6500    # Worse due to cross-VM networking
    }
}

# Calculate monthly costs
hours_per_month = 730
target_latency_ms = 2000

print(f"{'VM Configuration':<40} {'Cost/Mo':<12} {'Latency':<12} {'Meets Target?':<15}")
print("="*80)

results = []
for config_name, config in vm_configs.items():
    monthly_cost = config['hourly_cost'] * hours_per_month * config['min_vms_for_70b']
    latency = config['latency_estimate_ms']
    meets_target = "✅ Yes" if latency <= target_latency_ms else "❌ No"
    
    print(f"{config_name:<40} ${monthly_cost:>10,.0f}  {latency:>9,.0f}ms  {meets_target:<15}")
    
    results.append({
        'config': config_name,
        'monthly_cost': monthly_cost,
        'latency_ms': latency,
        'meets_target': latency <= target_latency_ms,
        'nvlink': config['nvlink'],
        'infiniband': config['infiniband']
    })

print("="*80)

# Find optimal choice
optimal = min(
    [r for r in results if r['meets_target']],
    key=lambda x: x['monthly_cost'],
    default=None
)

if optimal:
    print(f"\n🎯 Optimal Choice: {optimal['config']}")
    print(f"   - Monthly cost: ${optimal['monthly_cost']:,.0f}")
    print(f"   - Latency: {optimal['latency_ms']:.0f} ms (under {target_latency_ms} ms target)")
    print(f"   - NVLink: {'Yes' if optimal['nvlink'] else 'No'}")
    print(f"   - InfiniBand: {'Yes' if optimal['infiniband'] else 'No'}")
else:
    print(f"\n⚠️  No configuration meets <{target_latency_ms}ms latency target!")
    print("   → Need NVLink or faster interconnect")

# Cost vs performance trade-off
print(f"\n💡 Key Insights:")
print(f"   1. NVLink+IB (ND-series): ${vm_configs['ND96asr_v4 (8× A100 NVLink+IB)']['hourly_cost'] * hours_per_month:,.0f}/mo → 1.8s latency ✅")
print(f"   2. PCIe-only (NC-series): ${vm_configs['NC96ads_A100_v4 (4× A100 PCIe)']['hourly_cost'] * hours_per_month:,.0f}/mo → 5.2s latency ❌")
print(f"   3. Cost premium for NVLink: {(vm_configs['ND96asr_v4 (8× A100 NVLink+IB)']['hourly_cost'] / vm_configs['NC96ads_A100_v4 (4× A100 PCIe)']['hourly_cost'] - 1) * 100:.0f}%")
print(f"   4. Latency improvement: {vm_configs['NC96ads_A100_v4 (4× A100 PCIe)']['latency_estimate_ms'] / vm_configs['ND96asr_v4 (8× A100 NVLink+IB)']['latency_estimate_ms']:.1f}× faster")
print(f"\n   → Verdict: NVLink premium is worth it for latency-sensitive workloads")

print("\n⚠️  Remember to use Spot instances for 70-90% cost savings!")
print("   → Spot pricing: $27.20/hr → ~$8-10/hr (varies by region and demand)")

print("\n" + "="*80)
</VSCode.Cell>

<VSCode.Cell language="markdown">
## Topology Visualization

Visualize the GPU interconnect topology for Azure ND-series vs standard NC-series VMs. This helps understand the bandwidth hierarchy and communication patterns.
</VSCode.Cell>

<VSCode.Cell language="python">
# Cell 8: Topology visualization

print("=== GPU Topology Visualization ===\n")

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# Configure matplotlib
plt.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#1a1a2e'
plt.rcParams['axes.facecolor'] = '#1a1a2e'

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# ==================== Subplot 1: ND-Series (NVLink + InfiniBand) ====================
ax1.set_title('Azure ND96asr_v4 Topology\n8× A100 with NVLink + InfiniBand', 
              fontsize=13, fontweight='bold', pad=15)
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 10)
ax1.axis('off')

# Draw GPUs in a 2×4 grid (simulating NVLink mesh)
gpu_positions = [
    (1, 7), (3, 7), (5, 7), (7, 7),  # Top row
    (1, 4), (3, 4), (5, 4), (7, 4)   # Bottom row
]

for i, (x, y) in enumerate(gpu_positions):
    # GPU box
    gpu_box = FancyBboxPatch((x-0.4, y-0.4), 0.8, 0.8, 
                             boxstyle="round,pad=0.05", 
                             edgecolor='#15803d', facecolor='#15803d', 
                             linewidth=2, alpha=0.8)
    ax1.add_patch(gpu_box)
    ax1.text(x, y, f'A100\n#{i}', ha='center', va='center', 
             fontsize=9, fontweight='bold', color='white')
    
    # NVLink connections (green arrows)
    if i < 4:  # Top row to bottom row
        ax1.annotate('', xy=(x, y-0.4), xytext=(x, y-2.6),
                    arrowprops=dict(arrowstyle='<->', color='#15803d', lw=3, alpha=0.7))
    
    if i % 4 < 3:  # Horizontal connections
        ax1.annotate('', xy=(x+0.4, y), xytext=(x+1.6, y),
                    arrowprops=dict(arrowstyle='<->', color='#15803d', lw=3, alpha=0.7))

# NVLink label
ax1.text(4, 8.5, 'NVLink 3.0: 600 GB/s', ha='center', fontsize=11, 
         fontweight='bold', color='#15803d',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#15803d', alpha=0.3))

# InfiniBand (for multi-node)
ib_box = FancyBboxPatch((2, 0.5), 4, 1.5, boxstyle="round,pad=0.1",
                        edgecolor='#1d4ed8', facecolor='#1d4ed8', 
                        linewidth=2, alpha=0.5)
ax1.add_patch(ib_box)
ax1.text(4, 1.25, 'InfiniBand HDR\n200 Gb/s (25 GB/s)', ha='center', va='center',
         fontsize=10, fontweight='bold', color='white')

# Arrow from GPUs to IB
ax1.annotate('', xy=(4, 2), xytext=(4, 3.6),
            arrowprops=dict(arrowstyle='->', color='#1d4ed8', lw=2))

# ==================== Subplot 2: NC-Series (PCIe-only) ====================
ax2.set_title('Azure NC96ads_A100_v4 Topology\n4× A100 with PCIe Only', 
              fontsize=13, fontweight='bold', pad=15)
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')

# CPU in the center
cpu_box = FancyBboxPatch((3.5, 4.5), 3, 1.5, boxstyle="round,pad=0.1",
                        edgecolor='#b45309', facecolor='#b45309',
                        linewidth=2, alpha=0.7)
ax2.add_patch(cpu_box)
ax2.text(5, 5.25, 'CPU\n(PCIe Root)', ha='center', va='center',
         fontsize=10, fontweight='bold', color='white')

# GPUs around CPU
gpu_positions_nc = [(2, 8), (5, 8), (8, 8), (2, 2), (5, 2), (8, 2)][:4]

for i, (x, y) in enumerate(gpu_positions_nc):
    # GPU box
    gpu_box = FancyBboxPatch((x-0.4, y-0.4), 0.8, 0.8,
                             boxstyle="round,pad=0.05",
                             edgecolor='#b91c1c', facecolor='#b91c1c',
                             linewidth=2, alpha=0.8)
    ax2.add_patch(gpu_box)
    ax2.text(x, y, f'A100\n#{i}', ha='center', va='center',
             fontsize=9, fontweight='bold', color='white')
    
    # PCIe connection to CPU (red dashed)
    target_x = 5 if x < 5 else 5
    target_y = 5.25 if y > 5 else 4.75
    
    ax2.annotate('', xy=(x, y-0.4 if y > 5 else y+0.4), xytext=(target_x, target_y),
                arrowprops=dict(arrowstyle='<->', color='#b91c1c', lw=2, 
                              linestyle='--', alpha=0.7))

# PCIe label
ax2.text(5, 9.2, 'PCIe Gen4: 16 GB/s', ha='center', fontsize=11,
         fontweight='bold', color='#b91c1c',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#b91c1c', alpha=0.3))

# Warning label
ax2.text(5, 0.5, '⚠️  All GPU-to-GPU traffic\nrelayed through CPU', ha='center',
         fontsize=10, color='#b91c1c', fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#1a1a2e', 
                  edgecolor='#b91c1c', linewidth=2))

plt.tight_layout()
plt.savefig('img/azure_topology_comparison.png', dpi=150, facecolor='#1a1a2e')
plt.show()

# Summary table
print("\nTopology Comparison Summary:")
print("="*80)
print(f"{'Feature':<30} {'ND-Series (NVLink+IB)':<25} {'NC-Series (PCIe)':<25}")
print("="*80)
print(f"{'Within-node bandwidth':<30} {'600 GB/s (NVLink)':<25} {'16 GB/s (PCIe)':<25}")
print(f"{'Cross-node bandwidth':<30} {'25 GB/s (InfiniBand)':<25} {'N/A (Ethernet)':<25}")
print(f"{'GPU-to-GPU path':<30} {'Direct (NVLink)':<25} {'Via CPU (PCIe)':<25}")
print(f"{'Latency':<30} {'<1 μs (NVLink)':<25} {'5-10 μs (PCIe)':<25}")
print(f"{'Best for':<30} {'Tensor Parallelism':<25} {'Data Parallelism':<25}")
print(f"{'Cost premium':<30} {'+81%':<25} {'Baseline':<25}")
print("="*80)

print("\n✅ Topology visualization saved to: img/azure_topology_comparison.png")

print("\n" + "="*80)
print("Azure Supplement Complete!")
print("\nKey takeaways:")
print("  1. ND-series VMs have 40× faster GPU-to-GPU (NVLink vs PCIe)")
print("  2. InfiniBand enables multi-node scale-out (>8 GPUs)")
print("  3. Cost premium (81%) is justified for latency-sensitive workloads")
print("  4. Use NCCL tests to validate bandwidth in production")
print("  5. Always use Spot instances for 70-90% cost savings")
print("\n⚠️  CLEANUP: Delete Azure resources to avoid ongoing charges!")
print("   az group delete --name <resource-group> --yes --no-wait")
print("="*80)
</VSCode.Cell>
```